# TokenMM Markouts / Edge / PnL Demo

SQLite-first live telemetry notebook for Jeff. Core sections run entirely from durable local SQLite. Optional fill-time edge and extended-horizon sections appear only when a frozen FV extract is present.


## Scope And Methodology

- Canonical fill source: `execution_fill` from local SQLite
- Canonical durable markout source: `execution_markout` from local SQLite
- Durable live markouts are currently FV-only and deployed at `30s / 60s / 120s`
- Fills join to markouts on `trader_id + event_id`; `trade_id` is display-only context
- The directional PnL section is approximate current context from persisted balance / portfolio snapshots, not a true daily net ledger
- Optional local-market comparisons use `maker_mid` from the frozen FV stream as a proxy for local market because historical local BBO is not durably persisted elsewhere


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (CWD, CWD.parent, CWD.parent.parent, CWD.parent.parent.parent)
    if (path / 'pyproject.toml').exists()
)
FLUX_ROOT = REPO_ROOT / 'systems/flux'
for entry in (str(REPO_ROOT), str(FLUX_ROOT)):
    if entry not in sys.path:
        sys.path.insert(0, entry)

from research.tokenmm.telemetry_helpers import compute_extended_markouts_from_fv_stream
from research.tokenmm.telemetry_helpers import compute_fill_time_edge_rows
from research.tokenmm.telemetry_helpers import enrich_fills
from research.tokenmm.telemetry_helpers import load_fv_rows
from research.tokenmm.telemetry_helpers import load_sqlite_query
from research.tokenmm.telemetry_helpers import load_sqlite_table
from research.tokenmm.telemetry_helpers import latest_order_action_context
from research.tokenmm.telemetry_helpers import merge_fills_and_markouts
from research.tokenmm.telemetry_helpers import ms_to_utc
from research.tokenmm.telemetry_helpers import ns_to_utc
from research.tokenmm.telemetry_helpers import numeric
from research.tokenmm.telemetry_helpers import summarize_markouts
from research.tokenmm.telemetry_helpers import summarize_markouts_by_group
from research.tokenmm.telemetry_helpers import summarize_markouts_by_side

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 120
pd.options.display.width = 240

NOTEBOOK_ROOT = CWD
RESEARCH_ROOT = REPO_ROOT / 'research/tokenmm'
TELEMETRY_DIR = Path(os.environ.get('TOKENMM_TELEMETRY_DIR', '/var/lib/nautilus/telemetry/tokenmm'))
FV_EXTRACT_PATH = Path(os.environ.get('TOKENMM_FV_EXTRACT', str(RESEARCH_ROOT / 'data' / 'tokenmm_fv_extract.csv')))
CORE_HORIZONS = (30, 60, 120)
OPTIONAL_HORIZONS = (60, 120, 300, 1800, 3600)
DB_PATHS = {
    'execution_fill': TELEMETRY_DIR / 'fills.sqlite',
    'execution_markout': TELEMETRY_DIR / 'markouts.sqlite',
    'order_action': TELEMETRY_DIR / 'orders.sqlite',
    'quote_cycle': TELEMETRY_DIR / 'quote_cycles.sqlite',
    'balance_snapshot': TELEMETRY_DIR / 'balance_snapshots.sqlite',
    'portfolio_inventory': TELEMETRY_DIR / 'portfolio_inventory.sqlite',
}


def format_utc(series: pd.Series) -> pd.Series:
    values = pd.to_datetime(series, utc=True, errors='coerce')
    return values.dt.strftime('%Y-%m-%d %H:%M:%S UTC').fillna('n/a')


def format_number(series: pd.Series, digits: int = 2, suffix: str = '') -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    return values.map(lambda value: f'{value:.{digits}f}{suffix}' if pd.notna(value) else 'n/a')


def format_signed_number(series: pd.Series, digits: int = 2, suffix: str = '') -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    return values.map(lambda value: f'{value:+.{digits}f}{suffix}' if pd.notna(value) else 'n/a')


def weighted_mean(frame: pd.DataFrame, value_col: str, weight_col: str = 'fill_notional') -> float:
    values = pd.to_numeric(frame[value_col], errors='coerce')
    weights = pd.to_numeric(frame[weight_col], errors='coerce')
    mask = values.notna() & weights.notna() & (weights > 0)
    if not mask.any():
        return float('nan')
    return float((values[mask] * weights[mask]).sum() / weights[mask].sum())


DB_PATHS


{'execution_fill': PosixPath('/var/lib/nautilus/telemetry/tokenmm/fills.sqlite'),
 'execution_markout': PosixPath('/var/lib/nautilus/telemetry/tokenmm/markouts.sqlite'),
 'order_action': PosixPath('/var/lib/nautilus/telemetry/tokenmm/orders.sqlite'),
 'quote_cycle': PosixPath('/var/lib/nautilus/telemetry/tokenmm/quote_cycles.sqlite'),
 'balance_snapshot': PosixPath('/var/lib/nautilus/telemetry/tokenmm/balance_snapshots.sqlite'),
 'portfolio_inventory': PosixPath('/var/lib/nautilus/telemetry/tokenmm/portfolio_inventory.sqlite')}

## Data Loading

Core notebook cells load only the durable SQLite telemetry. The optional FV overlay reads a frozen CSV extract if it exists and otherwise leaves the extra sections out.


In [2]:
fills = enrich_fills(load_sqlite_table(DB_PATHS['execution_fill'], 'execution_fill'))
markouts = load_sqlite_table(DB_PATHS['execution_markout'], 'execution_markout')
markouts = markouts.loc[markouts['benchmark_name'].fillna('') == 'fv_market_mid'].copy()
markouts_joined = merge_fills_and_markouts(fills, markouts)

orders_recent = latest_order_action_context(
    load_sqlite_query(
        DB_PATHS['order_action'],
        '''
        SELECT trader_id, strategy_id, instrument_id, client_order_id, action_type, action_state,
               event_type, quote_cycle_id, reason_code, level_index, ts_event
        FROM order_action
        ORDER BY rowid DESC
        LIMIT 10000
        ''',
    )
)
quote_cycles_recent = load_sqlite_query(
    DB_PATHS['quote_cycle'],
    '''
    SELECT strategy_id, instrument_id, quote_cycle_id, quote_cycle_seq, quote_cycle_event,
           reason_code, trigger_source, ts_cycle_start_ns
    FROM quote_cycle
    ORDER BY rowid DESC
    LIMIT 10000
    ''',
)
recent_positions = load_sqlite_query(
    DB_PATHS['balance_snapshot'],
    '''
    SELECT strategy_id, exchange, instrument_id, side, signed_qty, quantity, avg_px_open, realized_pnl, ts_ms
    FROM flux_balance_snapshot_row
    WHERE kind = 'position'
    ORDER BY rowid DESC
    LIMIT 200
    ''',
)
recent_positions['signed_qty_num'] = numeric(recent_positions['signed_qty'])
recent_positions['quantity_num'] = numeric(recent_positions['quantity'])
recent_positions['avg_px_open_num'] = numeric(recent_positions['avg_px_open'])
recent_positions['realized_pnl_num'] = numeric(recent_positions['realized_pnl'].astype(str).str.split().str[0])
recent_positions['ts_utc'] = ms_to_utc(recent_positions['ts_ms'])
positions_current = (
    recent_positions.sort_values('ts_ms', ascending=False)
    .drop_duplicates(['strategy_id', 'instrument_id', 'side'])
    .loc[lambda frame: frame['signed_qty_num'].fillna(0).ne(0) | frame['quantity_num'].fillna(0).ne(0)]
    .copy()
)
portfolio_context = load_sqlite_query(
    DB_PATHS['portfolio_inventory'],
    '''
    SELECT portfolio_id, global_qty, global_qty_base, degraded, global_qty_complete,
           usable_component_count, expected_component_count, missing_required_json,
           stale_required_json, ts_ms
    FROM portfolio_inventory_snapshot
    ORDER BY rowid DESC
    LIMIT 1
    ''',
)
portfolio_context['ts_utc'] = ms_to_utc(portfolio_context['ts_ms'])

fv_extract_available = FV_EXTRACT_PATH.exists()
fv_rows = load_fv_rows(FV_EXTRACT_PATH) if fv_extract_available else pd.DataFrame()

fill_markout_status = (
    markouts_joined.groupby('fill_key', dropna=False)
    .agg(
        resolved_horizons=('resolution_status', lambda series: int((series == 'resolved').sum())),
        expired_horizons=('resolution_status', lambda series: int((series == 'expired').sum())),
        avg_resolved_markout_bps=('markout_bps_num', 'mean'),
    )
    .reset_index()
)

last_48h_cutoff_utc = pd.Timestamp.now(tz='UTC') - pd.Timedelta(hours=48)
fills_48h = fills.loc[fills['fill_ts_utc'] >= last_48h_cutoff_utc].copy()
fill_keys_48h = set(fills_48h['fill_key'].astype(str))
markouts_48h = markouts.loc[(markouts['trader_id'].astype(str) + '|' + markouts['event_id'].astype(str)).isin(fill_keys_48h)].copy()
markouts_joined_48h = markouts_joined.loc[markouts_joined['fill_key'].astype(str).isin(fill_keys_48h)].copy()

print(f'fills={len(fills)} markout_rows={len(markouts_joined)} fv_extract_available={fv_extract_available}')
print(f'fills_48h={len(fills_48h)} unique_fill_keys_48h={fills_48h['fill_key'].nunique()}')


fills=946 markout_rows=1317 fv_extract_available=False
fills_48h=443 unique_fill_keys_48h=443


## Sanity Checks

The first table is a quick telemetry heartbeat. The second table shows how much of the current fill set actually has durable markout rows. The horizon table confirms the deployed live horizons and whether rows resolved or expired.


In [3]:
telemetry_overview = pd.DataFrame(
    [
        {
            'table_name': 'execution_fill',
            'rows': len(fills),
            'latest_utc': fills['fill_ts_utc'].max(),
        },
        {
            'table_name': 'execution_markout',
            'rows': len(markouts_joined),
            'latest_utc': ms_to_utc(markouts_joined['target_ts_ms']).max(),
        },
        {
            'table_name': 'order_action_recent_slice',
            'rows': len(orders_recent),
            'latest_utc': ns_to_utc(orders_recent['ts_event']).max() if not orders_recent.empty else pd.NaT,
        },
        {
            'table_name': 'quote_cycle_recent_slice',
            'rows': len(quote_cycles_recent),
            'latest_utc': ns_to_utc(quote_cycles_recent['ts_cycle_start_ns']).max() if not quote_cycles_recent.empty else pd.NaT,
        },
        {
            'table_name': 'current_positions_slice',
            'rows': len(positions_current),
            'latest_utc': positions_current['ts_utc'].max() if not positions_current.empty else pd.NaT,
        },
    ]
)
telemetry_overview_display = telemetry_overview.assign(
    latest_utc=format_utc(pd.to_datetime(telemetry_overview['latest_utc'], utc=True, errors='coerce')),
)

coverage_overview = pd.DataFrame(
    [
        {
            'total_fills': int(fills['fill_key'].nunique()),
            'fills_with_markouts': int(markouts_joined['fill_key'].nunique()),
            'fill_coverage_pct': 100.0 * markouts_joined['fill_key'].nunique() / max(fills['fill_key'].nunique(), 1),
            'fv_extract_available': fv_extract_available,
            'fv_extract_path': str(FV_EXTRACT_PATH),
        }
    ]
)
coverage_overview_display = coverage_overview.assign(
    fill_coverage_pct=format_number(coverage_overview['fill_coverage_pct'], digits=1, suffix='%'),
)

horizon_coverage = (
    markouts_joined.groupby(['horizon_s', 'resolution_status'], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .sort_values('horizon_s')
)

telemetry_overview_display
coverage_overview_display
horizon_coverage


resolution_status,horizon_s,expired,resolved
0,30,108,332
1,60,107,332
2,120,106,332


## Last 48 Hours Presentation Table

This is the one compact table to present immediately. It is filtered to the last 48 hours of `execution_fill` timestamps, uses SQLite as the source of truth, and shows one row per `strategy_id + exchange` with both unweighted and notional-weighted FV markouts.


In [4]:
presentation_48h = summarize_markouts_by_group(
    fills=fills_48h,
    markouts=markouts_48h,
    group_cols=('strategy_id', 'venue'),
    horizons=CORE_HORIZONS,
).rename(columns={'venue': 'exchange'})
presentation_48h.insert(2, 'window_start_utc', last_48h_cutoff_utc)
presentation_48h.insert(3, 'window_end_utc', fills_48h['fill_ts_utc'].max() if not fills_48h.empty else pd.NaT)

presentation_48h_display = presentation_48h.assign(
    window_start_utc=format_utc(pd.to_datetime(presentation_48h['window_start_utc'], utc=True, errors='coerce')),
    window_end_utc=format_utc(pd.to_datetime(presentation_48h['window_end_utc'], utc=True, errors='coerce')),
    gross_notional=format_number(presentation_48h['gross_notional'], digits=2),
    avg_markout_bps_30s=format_signed_number(presentation_48h['avg_markout_bps_30s'], digits=2, suffix=' bps'),
    nw_markout_bps_30s=format_signed_number(presentation_48h['nw_markout_bps_30s'], digits=2, suffix=' bps'),
    avg_markout_bps_60s=format_signed_number(presentation_48h['avg_markout_bps_60s'], digits=2, suffix=' bps'),
    nw_markout_bps_60s=format_signed_number(presentation_48h['nw_markout_bps_60s'], digits=2, suffix=' bps'),
    avg_markout_bps_120s=format_signed_number(presentation_48h['avg_markout_bps_120s'], digits=2, suffix=' bps'),
    nw_markout_bps_120s=format_signed_number(presentation_48h['nw_markout_bps_120s'], digits=2, suffix=' bps'),
)[[
    'strategy_id', 'exchange', 'window_start_utc', 'window_end_utc', 'fill_count', 'gross_notional',
    'resolved_rows_30s', 'avg_markout_bps_30s', 'nw_markout_bps_30s',
    'resolved_rows_60s', 'avg_markout_bps_60s', 'nw_markout_bps_60s',
    'resolved_rows_120s', 'avg_markout_bps_120s', 'nw_markout_bps_120s',
]].sort_values(['gross_notional', 'strategy_id'], ascending=[False, True])
presentation_48h_display


,strategy_id,exchange,window_start_utc,window_end_utc,fill_count,gross_notional,resolved_rows_30s,avg_markout_bps_30s,nw_markout_bps_30s,resolved_rows_60s,avg_markout_bps_60s,nw_markout_bps_60s,resolved_rows_120s,avg_markout_bps_120s,nw_markout_bps_120s
6,plumeusdt_bybit_spot_makerv3-000,BITGET,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,7,83.63,2,-29.25 bps,-29.24 bps,1,-21.10 bps,-21.10 bps,4,+9.52 bps,+9.41 bps
9,plumeusdt_okx_perp_makerv3-000,BITGET,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,7,8.67,2,-15.86 bps,-15.86 bps,0,n/a,n/a,0,n/a,n/a
0,plumeusdt_bitget_perp_makerv3-000,BYBIT,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,14,75.18,1,+8.31 bps,+8.31 bps,1,+0.00 bps,+0.00 bps,1,+4.15 bps,+4.15 bps
10,plumeusdt_okx_perp_makerv3-000,BYBIT,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,51,61.75,32,-3.44 bps,-3.48 bps,33,-5.36 bps,-5.36 bps,27,-4.70 bps,-4.74 bps
1,plumeusdt_bitget_perp_makerv3-000,OKX,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,10,50.58,0,n/a,n/a,0,n/a,n/a,0,n/a,n/a
11,plumeusdt_okx_perp_makerv3-000,OKX,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,25,30.13,12,-8.28 bps,-8.30 bps,13,-8.00 bps,-7.99 bps,19,-9.22 bps,-9.13 bps
5,plumeusdt_bybit_perp_makerv3-000,OKX,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,23,264.53,14,+19.65 bps,+18.87 bps,14,+21.33 bps,+20.11 bps,6,+17.55 bps,+17.42 bps
7,plumeusdt_bybit_spot_makerv3-000,BYBIT,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,202,2327.25,193,+0.76 bps,+0.52 bps,194,+1.78 bps,+1.56 bps,197,+4.44 bps,+4.59 bps
2,plumeusdt_bitget_spot_makerv3-000,BITGET,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,1,2.99,1,+18.83 bps,+18.83 bps,1,+27.20 bps,+27.20 bps,1,+37.66 bps,+37.66 bps
3,plumeusdt_bitget_spot_makerv3-000,BYBIT,2026-03-11 17:20:34 UTC,2026-03-13 14:57:18 UTC,1,2.97,1,-25.25 bps,-25.25 bps,1,-27.36 bps,-27.36 bps,1,-29.46 bps,-29.46 bps


## Recent Fill Sample

This is a fast sanity slice of the most recent fills. It stays anchored on `execution_fill`, then pulls in the latest available order-action and quote-cycle context without changing fill identity.


In [5]:
recent_fill_sample = (
    fills.sort_values('fill_ts_ms', ascending=False)
    .merge(
        orders_recent[
            [
                'trader_id', 'client_order_id', 'action_type', 'action_state',
                'event_type', 'quote_cycle_id', 'reason_code', 'level_index'
            ]
        ],
        on=['trader_id', 'client_order_id'],
        how='left',
        suffixes=('', '_order'),
    )
    .merge(
        quote_cycles_recent[['quote_cycle_id', 'quote_cycle_event', 'trigger_source']].drop_duplicates('quote_cycle_id'),
        on='quote_cycle_id',
        how='left',
    )
    .merge(fill_markout_status, on='fill_key', how='left')
    .head(25)
)
recent_fill_sample_display = recent_fill_sample.assign(
    fill_ts_utc=format_utc(recent_fill_sample['fill_ts_utc']),
    fill_qty=format_number(recent_fill_sample['fill_qty_num'], digits=2),
    fill_px=format_number(recent_fill_sample['fill_px_num'], digits=8),
    fill_notional=format_number(recent_fill_sample['fill_notional'], digits=2),
    avg_resolved_markout_bps=format_signed_number(recent_fill_sample['avg_resolved_markout_bps'], digits=2, suffix=' bps'),
)[[
    'strategy_id', 'venue', 'symbol', 'order_side', 'fill_qty', 'fill_px', 'fill_notional',
    'trade_id', 'event_id', 'client_order_id', 'quote_cycle_id', 'action_type', 'quote_cycle_event',
    'resolved_horizons', 'expired_horizons', 'avg_resolved_markout_bps', 'fill_ts_utc'
]]
recent_fill_sample_display


,strategy_id,venue,symbol,order_side,fill_qty,fill_px,fill_notional,trade_id,event_id,client_order_id,quote_cycle_id,action_type,quote_cycle_event,resolved_horizons,expired_horizons,avg_resolved_markout_bps,fill_ts_utc
0,plumeusdt_bitget_spot_makerv3-000,BITGET,PLUMEUSDT,SELL,250.00,0.01195000,2.99,ad79d1b4-43a3-4dca-99b9-39d501c1e557,76ea17b0-77f0-4c5b-85e2-37806f27c7ab,O-20260313-145718-SPOT-000-5352,None,PLACE,NaN,3.0,0.0,+27.89 bps,2026-03-13 14:57:18 UTC
1,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,1000.00,0.01193000,11.93,2270000001368297838,a086edbf-d97e-4a83-be38-96dee0faf998,O-20260313-145619-SPOT-000-26455,None,NaN,NaN,3.0,0.0,-6.29 bps,2026-03-13 14:56:29 UTC
2,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,1000.00,0.01185000,11.85,2270000001368286134,45fda490-686a-47e2-8c7a-ecf3fa66fa73,O-20260313-145005-SPOT-000-25708,None,NaN,NaN,3.0,0.0,+33.76 bps,2026-03-13 14:50:11 UTC
3,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,1000.00,0.01186000,11.86,2270000001368286128,f39f29ef-db14-44b6-9c4a-90fff6e88fcf,O-20260313-145011-SPOT-000-25721,None,NaN,NaN,3.0,0.0,+25.30 bps,2026-03-13 14:50:11 UTC
4,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,1000.00,0.01187000,11.87,2270000001368286013,47e70e16-a65d-4c2d-aaf3-11cdc77e09c0,O-20260313-145001-SPOT-000-25702,None,NaN,NaN,3.0,0.0,+16.85 bps,2026-03-13 14:50:08 UTC
5,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,1000.00,0.01188000,11.88,2270000001368286006,5d7036ea-7c17-4809-8571-005273caf0c3,O-20260313-145003-SPOT-000-25705,None,NaN,NaN,3.0,0.0,+8.42 bps,2026-03-13 14:50:08 UTC
6,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,230.00,0.01212000,2.79,2270000001368211236,5e669d35-e5f5-4f23-b5a8-af04384fbd3c,O-20260313-140057-SPOT-000-22664,None,NaN,NaN,3.0,0.0,+4.81 bps,2026-03-13 14:01:02 UTC
7,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,770.00,0.01212000,9.33,2270000001368211233,6a0dacad-75c4-4697-8293-8cc5f64ab9f7,O-20260313-140057-SPOT-000-22664,None,NaN,NaN,3.0,0.0,+4.81 bps,2026-03-13 14:01:02 UTC
8,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,BUY,1000.00,0.01213000,12.13,2270000001368211220,e74dede2-71e4-4aac-aedc-258339bbf256,O-20260313-140057-SPOT-000-22662,None,NaN,NaN,3.0,0.0,-3.44 bps,2026-03-13 14:01:02 UTC
9,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,SELL,1000.00,0.01221000,12.21,2270000001368210390,65128935-08a5-4eaf-b83b-e912907f16f7,O-20260313-140043-SPOT-000-22622,None,NaN,NaN,3.0,0.0,+75.08 bps,2026-03-13 14:00:43 UTC


## FV Markouts By Horizon From SQLite

This summary is the durable local SQLite view. It only uses rows already persisted by the live markout actor, so it is the core source of truth for the demo.


In [6]:
topline_summary = summarize_markouts(fills=fills, markouts=markouts, horizons=CORE_HORIZONS)

topline_summary_display = topline_summary.assign(
    gross_notional=format_number(topline_summary['gross_notional'], digits=2),
    avg_markout_bps_30s=format_signed_number(topline_summary['avg_markout_bps_30s'], digits=2, suffix=' bps'),
    nw_markout_bps_30s=format_signed_number(topline_summary['nw_markout_bps_30s'], digits=2, suffix=' bps'),
    avg_markout_bps_60s=format_signed_number(topline_summary['avg_markout_bps_60s'], digits=2, suffix=' bps'),
    nw_markout_bps_60s=format_signed_number(topline_summary['nw_markout_bps_60s'], digits=2, suffix=' bps'),
    avg_markout_bps_120s=format_signed_number(topline_summary['avg_markout_bps_120s'], digits=2, suffix=' bps'),
    nw_markout_bps_120s=format_signed_number(topline_summary['nw_markout_bps_120s'], digits=2, suffix=' bps'),
)

horizon_curve = pd.DataFrame(
    {
        'horizon_s': list(CORE_HORIZONS),
        'avg_markout_bps': [
            topline_summary.loc[0, 'avg_markout_bps_30s'],
            topline_summary.loc[0, 'avg_markout_bps_60s'],
            topline_summary.loc[0, 'avg_markout_bps_120s'],
        ],
        'nw_markout_bps': [
            topline_summary.loc[0, 'nw_markout_bps_30s'],
            topline_summary.loc[0, 'nw_markout_bps_60s'],
            topline_summary.loc[0, 'nw_markout_bps_120s'],
        ],
    }
)
horizon_curve_display = horizon_curve.assign(
    avg_markout_bps=format_signed_number(horizon_curve['avg_markout_bps'], digits=2, suffix=' bps'),
    nw_markout_bps=format_signed_number(horizon_curve['nw_markout_bps'], digits=2, suffix=' bps'),
)

topline_summary_display
horizon_curve_display


,horizon_s,avg_markout_bps,nw_markout_bps
0,30,-0.26 bps,-0.03 bps
1,60,+2.08 bps,+2.77 bps
2,120,+4.52 bps,+5.91 bps


## Side Split (BUY / SELL)

This view keeps the same durable SQLite markout base and simply breaks it by fill side. Both simple averages and notional-weighted averages are shown.


In [7]:
side_summary = summarize_markouts_by_side(fills=fills, markouts=markouts, horizons=CORE_HORIZONS)
side_summary_display = side_summary.sort_values('order_side').assign(
    gross_notional=format_number(side_summary['gross_notional'], digits=2),
    avg_markout_bps_30s=format_signed_number(side_summary['avg_markout_bps_30s'], digits=2, suffix=' bps'),
    nw_markout_bps_30s=format_signed_number(side_summary['nw_markout_bps_30s'], digits=2, suffix=' bps'),
    avg_markout_bps_60s=format_signed_number(side_summary['avg_markout_bps_60s'], digits=2, suffix=' bps'),
    nw_markout_bps_60s=format_signed_number(side_summary['nw_markout_bps_60s'], digits=2, suffix=' bps'),
    avg_markout_bps_120s=format_signed_number(side_summary['avg_markout_bps_120s'], digits=2, suffix=' bps'),
    nw_markout_bps_120s=format_signed_number(side_summary['nw_markout_bps_120s'], digits=2, suffix=' bps'),
)
side_summary_display


,order_side,fill_count,gross_notional,resolved_rows_30s,avg_markout_bps_30s,nw_markout_bps_30s,resolved_rows_60s,avg_markout_bps_60s,nw_markout_bps_60s,resolved_rows_120s,avg_markout_bps_120s,nw_markout_bps_120s
0,BUY,221,2131.96,154,+4.92 bps,+5.09 bps,154,+3.63 bps,+4.16 bps,154,+2.68 bps,+3.92 bps
1,SELL,219,2062.30,178,-4.75 bps,-4.71 bps,178,+0.75 bps,+1.50 bps,178,+6.13 bps,+7.73 bps


## By Strategy / Venue / Symbol Pivots

The live footprint is small enough that one compact table grouped by `strategy_id + venue + symbol` is easier to read than a large pivot matrix. This keeps counts, notionals, and the three durable markout horizons in one place.


In [8]:
group_summary = summarize_markouts_by_group(
    fills=fills,
    markouts=markouts,
    group_cols=('strategy_id', 'venue', 'symbol'),
    horizons=CORE_HORIZONS,
).sort_values(['gross_notional', 'strategy_id'], ascending=[False, True])

group_summary_display = group_summary.assign(
    gross_notional=format_number(group_summary['gross_notional'], digits=2),
    avg_markout_bps_30s=format_signed_number(group_summary['avg_markout_bps_30s'], digits=2, suffix=' bps'),
    nw_markout_bps_30s=format_signed_number(group_summary['nw_markout_bps_30s'], digits=2, suffix=' bps'),
    avg_markout_bps_60s=format_signed_number(group_summary['avg_markout_bps_60s'], digits=2, suffix=' bps'),
    nw_markout_bps_60s=format_signed_number(group_summary['nw_markout_bps_60s'], digits=2, suffix=' bps'),
    avg_markout_bps_120s=format_signed_number(group_summary['avg_markout_bps_120s'], digits=2, suffix=' bps'),
    nw_markout_bps_120s=format_signed_number(group_summary['nw_markout_bps_120s'], digits=2, suffix=' bps'),
)
group_summary_display


,strategy_id,venue,symbol,fill_count,gross_notional,resolved_rows_30s,avg_markout_bps_30s,nw_markout_bps_30s,resolved_rows_60s,avg_markout_bps_60s,nw_markout_bps_60s,resolved_rows_120s,avg_markout_bps_120s,nw_markout_bps_120s
3,plumeusdt_bybit_spot_makerv3-000,BYBIT,PLUMEUSDT,222,2573.63,207,+0.83 bps,+0.62 bps,207,+2.13 bps,+1.94 bps,207,+4.62 bps,+4.78 bps
2,plumeusdt_bybit_perp_makerv3-000,BYBIT,PLUMEUSDT,132,1454.38,76,-0.26 bps,-1.60 bps,76,+7.00 bps,+5.82 bps,76,+10.99 bps,+10.06 bps
4,plumeusdt_okx_perp_makerv3-000,OKX,PLUME-USDT,70,85.11,46,-5.25 bps,-5.28 bps,46,-6.10 bps,-6.11 bps,46,-6.57 bps,-6.55 bps
0,plumeusdt_bitget_perp_makerv3-000,BITGET,PLUMEUSDT,14,75.18,1,+8.31 bps,+8.31 bps,1,+0.00 bps,+0.00 bps,1,+4.15 bps,+4.15 bps
1,plumeusdt_bitget_spot_makerv3-000,BITGET,PLUMEUSDT,2,5.96,2,-3.21 bps,-3.15 bps,2,-0.08 bps,+0.00 bps,2,+4.10 bps,+4.20 bps


## Directional PnL Context

This is intentionally a small, caveated current-context section. It uses the latest persisted position rows we can pull quickly plus the latest shared portfolio inventory snapshot. It is **not** a true daily or net PnL ledger.


In [9]:
directional_context = pd.DataFrame(
    [
        {
            'position_rows': len(positions_current),
            'latest_position_utc': positions_current['ts_utc'].max() if not positions_current.empty else pd.NaT,
            'sum_realized_pnl_snapshot': positions_current['realized_pnl_num'].sum(),
            'net_signed_qty': positions_current['signed_qty_num'].sum(),
            'portfolio_global_qty': portfolio_context.loc[0, 'global_qty'] if not portfolio_context.empty else None,
            'portfolio_degraded': bool(portfolio_context.loc[0, 'degraded']) if not portfolio_context.empty else None,
            'usable_components': (
                f"{int(portfolio_context.loc[0, 'usable_component_count'])}/{int(portfolio_context.loc[0, 'expected_component_count'])}"
                if not portfolio_context.empty else 'n/a'
            ),
        }
    ]
)

directional_context_display = directional_context.assign(
    latest_position_utc=format_utc(pd.to_datetime(directional_context['latest_position_utc'], utc=True, errors='coerce')),
    sum_realized_pnl_snapshot=format_signed_number(directional_context['sum_realized_pnl_snapshot'], digits=2),
    net_signed_qty=format_signed_number(directional_context['net_signed_qty'], digits=2),
)
positions_current_display = positions_current.assign(
    ts_utc=format_utc(positions_current['ts_utc']),
    signed_qty=format_signed_number(positions_current['signed_qty_num'], digits=2),
    quantity=format_number(positions_current['quantity_num'], digits=2),
    avg_px_open=format_number(positions_current['avg_px_open_num'], digits=8),
    realized_pnl=format_signed_number(positions_current['realized_pnl_num'], digits=2),
)[[
    'strategy_id', 'exchange', 'instrument_id', 'side', 'signed_qty', 'quantity', 'avg_px_open', 'realized_pnl', 'ts_utc'
]].sort_values(['strategy_id', 'instrument_id', 'side'])
portfolio_context_display = portfolio_context.assign(
    ts_utc=format_utc(portfolio_context['ts_utc']),
)[[
    'portfolio_id', 'global_qty', 'global_qty_base', 'degraded', 'global_qty_complete',
    'usable_component_count', 'expected_component_count', 'missing_required_json',
    'stale_required_json', 'ts_utc'
]]

directional_context_display
positions_current_display
portfolio_context_display


,portfolio_id,global_qty,global_qty_base,degraded,global_qty_complete,usable_component_count,expected_component_count,missing_required_json,stale_required_json,ts_utc
0,tokenmm,46022.49720956,46022.49720956,0,1,6,7,[],[],2026-03-13 17:20:34 UTC


## Optional Fill-Time Edge / Extended Horizons

These sections only render when a frozen FV extract is present. The notebook never depends on live Redis during the demo. `maker_mid` is explicitly treated as a proxy for local market in these optional tables.


In [10]:
optional_extract_status = pd.DataFrame(
    [
        {
            'fv_extract_available': fv_extract_available,
            'fv_extract_path': str(FV_EXTRACT_PATH),
            'note': (
                'Optional edge and extended-horizon sections are enabled.'
                if fv_extract_available else
                'Frozen FV extract not found. Optional sections are intentionally omitted.'
            ),
        }
    ]
)
optional_extract_status

if fv_extract_available and not fv_rows.empty:
    fill_lookup_base = fills[['trader_id', 'event_id', 'strategy_id', 'order_side', 'fill_px_num', 'fill_qty_num', 'fill_notional', 'fill_ts_ms']].copy()
    edge_rows = compute_fill_time_edge_rows(fills=fill_lookup_base, fv_rows=fv_rows)
    edge_rows = edge_rows.merge(
        fill_lookup_base[['trader_id', 'event_id', 'fill_notional']].drop_duplicates(),
        on=['trader_id', 'event_id'],
        how='left',
    )
    edge_summary = (
        edge_rows.groupby('benchmark_name', dropna=False)
        .apply(
            lambda frame: pd.Series(
                {
                    'fill_count': int(frame['event_id'].nunique()),
                    'avg_edge_bps': frame['edge_bps'].mean(),
                    'nw_edge_bps': weighted_mean(frame, 'edge_bps'),
                    'avg_lag_ms': numeric(frame['lag_ms']).mean(),
                    'p95_lag_ms': numeric(frame['lag_ms']).quantile(0.95),
                }
            )
        )
        .reset_index()
        .sort_values('benchmark_name')
    )
    edge_summary_display = edge_summary.assign(
        avg_edge_bps=format_signed_number(edge_summary['avg_edge_bps'], digits=2, suffix=' bps'),
        nw_edge_bps=format_signed_number(edge_summary['nw_edge_bps'], digits=2, suffix=' bps'),
        avg_lag_ms=format_number(edge_summary['avg_lag_ms'], digits=1, suffix=' ms'),
        p95_lag_ms=format_number(edge_summary['p95_lag_ms'], digits=1, suffix=' ms'),
    )
    edge_rows_display = edge_rows.assign(
        edge_abs=format_number(edge_rows['edge_abs'], digits=8),
        edge_bps=format_signed_number(edge_rows['edge_bps'], digits=2, suffix=' bps'),
        benchmark_px=format_number(edge_rows['benchmark_px'], digits=8),
        lag_ms=format_number(edge_rows['lag_ms'], digits=0, suffix=' ms'),
    )[[
        'benchmark_name', 'trader_id', 'event_id', 'strategy_id', 'fill_ts_ms',
        'benchmark_ts_ms', 'lag_ms', 'benchmark_px', 'edge_abs', 'edge_bps', 'status'
    ]].head(20)

    extended_rows = compute_extended_markouts_from_fv_stream(
        fills=fill_lookup_base,
        fv_rows=fv_rows,
        horizons_s=OPTIONAL_HORIZONS,
        benchmark_names=('fv', 'maker_mid'),
    )
    extended_rows = extended_rows.merge(
        fill_lookup_base[['trader_id', 'event_id', 'fill_notional']].drop_duplicates(),
        on=['trader_id', 'event_id'],
        how='left',
    )
    extended_summary = (
        extended_rows.groupby(['benchmark_name', 'horizon_s'], dropna=False)
        .apply(
            lambda frame: pd.Series(
                {
                    'fill_count': int(frame['event_id'].nunique()),
                    'avg_markout_bps': frame['markout_bps'].mean(),
                    'nw_markout_bps': weighted_mean(frame, 'markout_bps'),
                    'avg_lag_ms': numeric(frame['lag_ms']).mean(),
                }
            )
        )
        .reset_index()
        .sort_values(['benchmark_name', 'horizon_s'])
    )
    extended_summary_display = extended_summary.assign(
        avg_markout_bps=format_signed_number(extended_summary['avg_markout_bps'], digits=2, suffix=' bps'),
        nw_markout_bps=format_signed_number(extended_summary['nw_markout_bps'], digits=2, suffix=' bps'),
        avg_lag_ms=format_number(extended_summary['avg_lag_ms'], digits=1, suffix=' ms'),
    )

    edge_summary_display
    edge_rows_display
    extended_summary_display


## Takeaways / Caveats

- Durable demo truth comes from local SQLite `execution_fill` + `execution_markout`
- Durable live markouts are currently FV-only and limited to `30s / 60s / 120s`
- Side splits and strategy / venue / symbol pivots above are therefore durable and demo-safe
- Optional edge / local-market sections require a frozen FV extract; if absent, the notebook omits them instead of hitting live Redis
- When those optional sections are present, `maker_mid` is explicitly a proxy for local market rather than a separately persisted local BBO history
- The PnL section is current-mark / current-position context only and should not be described as true daily or net firm PnL
